# Soft segment-Viterbi: training
Source: the `ssv` package of the paper's code release (embedded below).

In [ ]:
import os
os.makedirs('/kaggle/working/ssv', exist_ok=True)


In [ ]:
%%writefile /kaggle/working/ssv/__init__.py
"""Soft segment-Viterbi (SSV) for true vertical thickness (TVT) prediction in horizontal wells.

A learned segment cost and a learned transition (momentum) cost are trained end-to-end through a
differentiable (soft-min) segment-Viterbi over a lattice of straight structural segments.
"""
__version__ = "1.0.0"


In [ ]:
%%writefile /kaggle/working/ssv/baselines.py
"""Reference predictors evaluated with the same protocol as the model (prediction zone of every training well).

    python -m ssv.baselines --out baselines.pkl [--hand]   (--hand adds the hand-crafted-cost Viterbi; slower)
"""
import argparse
import math
import pickle
import numpy as np
import torch
from .config import Config
from .data import load_split, folds
from .features import prep, to_torch, gr_stats, lattice_start, Lattice
from .model import SSVNet, fit_stats, decode, prediction_rows
from . import neighbours


def hold_tvt(d):
    """Last known TVT, held constant."""
    return np.full(len(d["MD"]) - d["K"], d["TVT_input"][d["K"] - 1])


def hold_s_trend(d):
    """Layers parallel to the trajectory trend: TVT = TVT_K + z_det - z_det(K), z_det = Z_lin - Z."""
    K, md, Z = d["K"], d["MD"], d["Z"]
    mz, cz = np.polyfit(md[K:], Z[K:], 1)
    zdet = (mz * md + cz) - Z
    return d["TVT_input"][K - 1] + zdet[K:] - zdet[K - 1]


def hold_s_flat(d):
    """Flat layers: S = TVT + Z constant."""
    K = d["K"]
    return d["TVT_input"][K - 1] - (d["Z"][K:] - d["Z"][K - 1])


def lattice_oracle(d, cfg):
    """Best lattice path given the truth (min squared error): the resolution limit of the lattice."""
    D, KC, MV = cfg.chunk, cfg.kc, cfg.mv
    K, N = d["K"], len(d["MD"])
    s, r = lattice_start(K, N, D)
    md, Z = d["MD"], d["Z"]
    mz, cz = np.polyfit(md[s:], Z[s:], 1)
    zd = (mz * md + cz) - Z
    off = d["TVT_input"][s] + zd[s:] - zd[s]
    y = np.r_[d["TVT_input"][s:K], d["TVT"][K:]] - off
    n = len(y)
    C = math.ceil((n - 1) / D)
    y = np.r_[y, np.full(C * D + 1 - n, y[-1])]
    wt = np.zeros(len(y)); wt[r:n] = 1
    A = np.arange(-KC, KC + 1, dtype=float)[:, None]
    M = np.arange(-MV, MV + 1, dtype=float)[None, :]
    INF = 1e18
    al = np.full(2 * KC + 1, INF); al[KC] = 0
    bps = []
    for c in range(C):
        t = np.arange(D + (c == C - 1)); rows = c * D + t; tau = t / D; w = wt[rows]; yy = y[rows]
        s0, s1, s2 = w.sum(), (w * tau).sum(), (w * tau * tau).sum()
        y0, y1, y2 = (w * yy).sum(), (w * tau * yy).sum(), (w * yy * yy).sum()
        cost = A * A * s0 + 2 * A * M * s1 + M * M * s2 - 2 * A * y0 - 2 * M * y1 + y2
        tot = al[:, None] + cost
        new = np.full(2 * KC + 1, INF); arg = np.zeros(2 * KC + 1, int)
        for k in range(2 * MV + 1):
            src = np.arange(2 * KC + 1); dst = src + k - MV; ok = (dst >= 0) & (dst <= 2 * KC)
            v = tot[src[ok], k]; b = v < new[dst[ok]]
            new[dst[ok][b]] = v[b]; arg[dst[ok][b]] = src[ok][b]
        al = new; bps.append(arg)
    k = int(al.argmin()); anc = [k]
    for c in range(C - 1, -1, -1):
        k = bps[c][k]; anc.append(k)
    anc = np.array(anc[::-1], float) - KC
    f = np.arange(D) / D
    path = np.r_[np.concatenate([anc[i] + (anc[i + 1] - anc[i]) * f for i in range(C)]), anc[-1]][:n]
    return (path + off)[r:]


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="baselines.pkl")
    ap.add_argument("--nb-cache", default="nb_train.pkl")
    ap.add_argument("--data", default=None)
    ap.add_argument("--hand", action="store_true")
    a = ap.parse_args(argv)
    cfg = Config()
    W = load_split("train", a.data)
    NB = neighbours.train_cache(W, a.nb_cache)
    names = sorted(W)
    f = folds(names)
    res = {}
    for w in names:
        d = W[w]
        res[w] = dict(hold_tvt=hold_tvt(d), hold_s_flat=hold_s_flat(d), hold_s_trend=hold_s_trend(d),
                      field=NB[w]["path"].astype(float), oracle=lattice_oracle(d, cfg))
    if a.hand:   # Viterbi with the clipped squared GR misfit as the only cost (first order, untrained network)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        hc = Config(momentum=False)
        for k in range(5):
            tr = [w for w, fk in zip(names, f) if fk != k]
            va = [w for w, fk in zip(names, f) if fk == k]
            gmu, gsd = gr_stats({w: W[w] for w in tr})
            lat = Lattice(hc, dev)
            Ptr = [to_torch(prep(w, W[w], NB[w], hc, gmu, gsd), dev) for w in tr[::4]]
            st = fit_stats(Ptr, lat, hc)
            net = SSVNet(hc, st).to(dev).eval()
            with torch.no_grad():
                for i in range(0, len(va), 4):
                    ps = [to_torch(prep(w, W[w], NB[w], hc, gmu, gsd), dev) for w in va[i:i + 4]]
                    for t, p in zip(decode(net, ps, lat, hard=True), ps):
                        res[p["w"]]["hand_viterbi"] = prediction_rows(t, p).cpu().numpy()
            print("hand-cost Viterbi fold", k, flush=True)
    pickle.dump(res, open(a.out, "wb"))
    from .evaluate import pooled_rmse
    for key in res[names[0]]:
        print(f"{key:14s} {pooled_rmse({w: res[w] for w in names}, W, key):.3f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ssv/config.py
"""All tunable settings in one place. Every value that is not a pure implementation detail is listed here,
with the analysis that motivates it (see the paper, Sections 3-4)."""
from dataclasses import dataclass, asdict, fields
import json


@dataclass
class Config:
    # ---------------- lattice (Sec. 4.1) ----------------
    chunk: int = 80            # rows (= ft of MD) per straight segment; 89% of the interpreted path lies in pieces >= 80 ft
    kc: int = 100              # anchors on each side of the trend line, 1 ft apart (lattice spans +-100 ft)
    max_slope: float = 0.05    # |dS/dMD| relative to the trajectory trend; covers 98.6% of the interpreted path length
    # ---------------- inputs (Sec. 4.2) ----------------
    pts_per_chunk: int = 16    # samples of the segment line per chunk
    ext_back: int = 2          # chunks of context before the segment (line extended backwards)
    ext_fwd: int = 2           # chunks of context after the segment (line extended forwards)
    gr_box: int = 5            # rows of box smoothing of the lateral GR after the low-pass filter
    tw_step: float = 0.25      # ft, resampling step of the typewell
    seen_blend: float = 0.7    # weight of the well's own known section in the reference GR(TVT) curve
    seen_window: float = 100.0 # ft of TVT around the last known TVT used for the own-section reference
    misfit_clip: float = -1.0  # clip of the squared GR misfit in standardized units; <0 -> value from the stats file
    scaling: str = "data"      # 'data': standardize with training statistics; 'legacy': the earlier fixed scales (ablation)
    # ---------------- model (Sec. 4.3) ----------------
    hidden: int = 32           # conv channels of the segment-cost network
    mlp: int = 64              # hidden units of the segment-cost head
    momentum: bool = True      # second-order lattice with a learned transition cost
    position: bool = True      # distance from the last known row as an input
    neighbour: bool = True     # neighbour structure-field inputs
    seen: bool = True          # blend the own known section into the reference curve
    logT0: float = -3.0        # initial log temperature of the soft-min
    # ---------------- training (Sec. 4.4) ----------------
    epochs: int = 15
    batch_wells: int = 4
    lr: float = 1e-3
    lr_min_frac: float = 0.05  # cosine schedule floor (fraction of lr)
    weight_decay: float = 1e-4
    ema: float = 0.99
    grad_clip: float = 5.0
    loss_norm: str = "global"  # 'global': every predicted row has the same weight in every batch (unbiased for the
                               #            row-pooled RMSE); 'batch': divide by the rows of the batch; 'well': mean per well
    nb_dropout: float = 0.0    # probability of hiding the neighbour inputs of a training well
    seed: int = 0
    # ---------------- cross-validation ----------------
    n_folds: int = 5
    fold: int = 0              # validation fold; -1 = train on all wells
    fold_seed: int = 42
    tag: str = "ssv"

    @property
    def mv(self) -> int:       # largest anchor move per chunk
        return int(round(self.max_slope * self.chunk))

    def to_json(self) -> str:
        return json.dumps(asdict(self), sort_keys=True)

    @classmethod
    def from_dict(cls, d: dict) -> "Config":
        names = {f.name for f in fields(cls)}
        unknown = set(d) - names
        if unknown:
            raise ValueError(f"unknown config keys: {sorted(unknown)}")
        return cls(**d)

    @classmethod
    def from_json(cls, s: str) -> "Config":
        return cls.from_dict(json.loads(s))


In [ ]:
%%writefile /kaggle/working/ssv/data.py
"""Reading the competition files and the cross-validation split."""
import glob
import os
import numpy as np
import pandas as pd

KAGGLE_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"


def data_dir() -> str:
    """Competition folder (holds train/, test/, sample_submission.csv)."""
    if os.environ.get("ROGII_DATA"):
        return os.environ["ROGII_DATA"]
    if os.path.exists(KAGGLE_DIR):
        return KAGGLE_DIR
    for c in ["../data", "../../data", "data"]:
        if os.path.exists(os.path.join(c, "train")):
            return c
    raise FileNotFoundError("set ROGII_DATA to the competition folder")


def load_split(split: str, base: str = None) -> dict:
    """Return {well: dict of numpy arrays}. K = number of rows with known TVT_input (the known section)."""
    base = base or data_dir()
    wells = {}
    for f in sorted(glob.glob(os.path.join(base, split, "*__horizontal_well.csv"))):
        w = os.path.basename(f).split("__")[0]
        hw = pd.read_csv(f)
        tw = pd.read_csv(os.path.join(base, split, f"{w}__typewell.csv"))
        d = {c: hw[c].to_numpy(float) for c in ["MD", "X", "Y", "Z", "GR", "TVT_input"]}
        if "TVT" in hw:
            d["TVT"] = hw["TVT"].to_numpy(float)
        d["tw_TVT"] = tw["TVT"].to_numpy(float)
        d["tw_GR"] = tw["GR"].to_numpy(float)
        known = np.flatnonzero(np.isfinite(d["TVT_input"]))
        d["K"] = int(known[-1]) + 1
        wells[w] = d
    return wells


def folds(names, n_folds=5, seed=42) -> np.ndarray:
    """Random split by well: sorted names, fixed permutation, fold = position mod n_folds."""
    names = sorted(names)
    perm = np.random.RandomState(seed).permutation(len(names))
    f = np.zeros(len(names), int)
    f[perm] = np.arange(len(names)) % n_folds
    return f


def split_names(names, fold, n_folds=5, seed=42):
    names = sorted(names)
    if fold < 0:
        return names, []
    f = folds(names, n_folds, seed)
    return [w for w, k in zip(names, f) if k != fold], [w for w, k in zip(names, f) if k == fold]


In [ ]:
%%writefile /kaggle/working/ssv/dp.py
"""Second-order (momentum) segment-Viterbi on the anchor lattice: soft (forward-backward with soft-min) and hard.

State at chunk boundary c: (anchor a in [-KC..KC], move m in [-MV..MV] of the chunk that ends at c).
Path cost  = sum_c cost[c, a_c, m_c] + tau[c, m_{c-1}, m_c],   with tau[0] replaced by tau0[m_0]
             (tau0 conditions the first move on the move of the known section just before the lattice).
cost[b, c, a, m] is indexed by the START anchor a of chunk c and its move m; the segment ends at a + m.
Disallowed transitions carry the cost BIG.
"""
import torch
import torch.nn.functional as F

BIG = 1e4


def softmin(x, T, dim):
    return -T * torch.logsumexp(-x / T, dim)


def _shift_in(x, MV, NA):
    """x[B, NA, NM] indexed by (start anchor, move) -> y[B, NA, NM] indexed by (end anchor a+m, move)."""
    B, _, NM = x.shape
    xp = F.pad(x, (0, 0, MV, MV), value=BIG)
    return torch.stack([xp[:, 2 * MV - k:2 * MV - k + NA, k] for k in range(NM)], 2)


def _shift_out(y, MV, NA):
    """Inverse of _shift_in: (end anchor, move) -> (start anchor, move)."""
    B, _, NM = y.shape
    yp = F.pad(y, (0, 0, MV, MV), value=BIG)
    return torch.stack([yp[:, k:k + NA, k] for k in range(NM)], 2)


def forward_backward(cost, tau, tau0, T, KC, MV, return_marginals=False):
    """Expected anchor (posterior mean under exp(-cost/T)) at every boundary 0..C.

    cost [B, C, NA, NM], tau [B, C, NM, NM] (entry [c, p, m] used for c >= 1), tau0 [B, NM], T > 0.
    Returns S [B, C+1] (S[:, 0] = 0) and, optionally, the anchor marginals [B, C, NA].
    """
    B, C, NA, NM = cost.shape
    dev, dt = cost.device, cost.dtype
    AV = torch.arange(-KC, KC + 1, dtype=dt, device=dev)
    a0 = torch.full((B, NA), BIG, device=dev, dtype=dt)
    a0[:, KC] = 0
    al = [_shift_in(a0[:, :, None] + tau0[:, None, :] + cost[:, 0], MV, NA)]
    for c in range(1, C):
        bm = softmin(al[-1][:, :, :, None] + tau[:, c][:, None], T, 2)
        al.append(_shift_in(bm + cost[:, c], MV, NA))
    be = [torch.zeros(B, NA, NM, device=dev, dtype=dt)]
    for c in range(C - 1, 0, -1):
        g = _shift_out(be[0], MV, NA) + cost[:, c]
        be.insert(0, softmin(tau[:, c][:, None] + g[:, :, None, :], T, 3))
    S = [torch.zeros(B, device=dev, dtype=dt)]
    marg = []
    for c in range(C):
        p = torch.softmax((-(al[c] + be[c]) / T).reshape(B, -1), 1).reshape(B, NA, NM).sum(2)
        marg.append(p)
        S.append((p * AV).sum(1))
    S = torch.stack(S, 1)
    return (S, torch.stack(marg, 1)) if return_marginals else S


@torch.no_grad()
def viterbi(cost, tau, tau0, KC, MV):
    """Minimum-cost anchor path [B, C+1] (anchor values, S[:, 0] = 0)."""
    B, C, NA, NM = cost.shape
    dev = cost.device
    a0 = torch.full((B, NA), BIG, device=dev, dtype=cost.dtype)
    a0[:, KC] = 0
    al = _shift_in(a0[:, :, None] + tau0[:, None, :] + cost[:, 0], MV, NA)
    bps = []
    for c in range(1, C):
        v, ix = (al[:, :, :, None] + tau[:, c][:, None]).min(2)
        bps.append(ix)
        al = _shift_in(v + cost[:, c], MV, NA)
    flat = al.reshape(B, -1).argmin(1)
    a, m = flat // NM, flat % NM
    anc = [a]
    for c in range(C - 1, 0, -1):
        a = a - (m - MV)
        anc.append(a)
        m = bps[c - 1].gather(1, a[:, None, None].expand(-1, 1, NM))[:, 0].gather(1, m[:, None])[:, 0]
    anc.append(torch.full_like(a, KC))
    return torch.stack(anc[::-1], 1).to(cost.dtype) - KC


def brute_force(cost, tau, tau0, T, KC, MV):
    """Enumerate all move sequences (tiny lattices only). Returns (soft expectation, best path) per batch item."""
    import itertools
    import math
    B, C, NA, NM = cost.shape
    out = []
    for b in range(B):
        Z, E, best = 0.0, torch.zeros(C + 1, dtype=torch.float64), (float("inf"), None)
        for ms in itertools.product(range(NM), repeat=C):
            a, tot, path, ok = 0, float(tau0[b, ms[0]]), [0], True
            for c, m in enumerate(ms):
                if c > 0:
                    tot += float(tau[b, c, ms[c - 1], m])
                tot += float(cost[b, c, a + KC, m])
                a = a + m - MV
                path.append(a)
                ok &= abs(a) <= KC
            if not ok or tot >= BIG / 2:
                continue
            w = math.exp(-tot / T)
            Z += w
            E += w * torch.tensor(path, dtype=torch.float64)
            if tot < best[0]:
                best = (tot, path)
        out.append(((E / Z).float(), torch.tensor(best[1], dtype=torch.float32)))
    return out


In [ ]:
%%writefile /kaggle/working/ssv/evaluate.py
"""Metrics: row-pooled RMSE (the competition metric), per-well RMSE, error concentration, error along the lateral.

    python -m ssv.evaluate runs/f*s0/val.pkl            # out-of-fold summary of one seed
"""
import argparse
import glob
import math
import pickle
import numpy as np
from .data import load_split


def pooled_rmse(pred: dict, wells: dict, key=None):
    se = n = 0.0
    for w, t in pred.items():
        t = t[key] if key else t
        y = wells[w]["TVT"][wells[w]["K"]:]
        m = np.isfinite(y)
        se += float(((t - y)[m] ** 2).sum())
        n += m.sum()
    return math.sqrt(se / n)


def per_well(pred: dict, wells: dict, key=None):
    out = {}
    for w, t in pred.items():
        t = t[key] if key else t
        y = wells[w]["TVT"][wells[w]["K"]:]
        m = np.isfinite(y)
        out[w] = dict(rmse=float(np.sqrt(np.mean((t - y)[m] ** 2))), sse=float(((t - y)[m] ** 2).sum()), n=int(m.sum()))
    return out


def concentration(pw: dict, drops=(0, 1, 3, 5, 10, 20, 50)):
    """RMSE of the remaining wells after removing the k worst (by squared error), and the share removed."""
    sse = np.array(sorted((v["sse"] for v in pw.values()), reverse=True))
    order = sorted(pw.values(), key=lambda v: -v["sse"])
    n = np.array([v["n"] for v in order])
    tot = sse.sum()
    return [(k, math.sqrt(sse[k:].sum() / n[k:].sum()), sse[:k].sum() / tot) for k in drops]


def by_position(pred: dict, wells: dict, key=None, bins=10):
    """RMSE by decile of relative position in the prediction zone."""
    se, n = np.zeros(bins), np.zeros(bins)
    for w, t in pred.items():
        t = t[key] if key else t
        y = wells[w]["TVT"][wells[w]["K"]:]
        b = np.minimum((np.arange(len(y)) * bins) // len(y), bins - 1)
        m = np.isfinite(y)
        np.add.at(se, b[m], (t - y)[m] ** 2)
        np.add.at(n, b[m], 1)
    return np.sqrt(se / n)


def load_oof(paths):
    out = {}
    for p in paths:
        out.update(pickle.load(open(p, "rb")))
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("val", nargs="+", help="val.pkl files (one per fold)")
    ap.add_argument("--data", default=None)
    a = ap.parse_args(argv)
    W = load_split("train", a.data)
    paths = [p for g in a.val for p in glob.glob(g)]
    oof = load_oof(paths)
    print(f"{len(oof)} wells from {len(paths)} files")
    for k in ("soft", "hard"):
        pw = per_well(oof, W, k)
        print(f"{k}: pooled RMSE {pooled_rmse(oof, W, k):.3f}")
        print("   worst removed / RMSE of rest / share of squared error:",
              [(c, round(r, 2), round(s, 2)) for c, r, s in concentration(pw)])
        print("   RMSE by position decile:", by_position(oof, W, k).round(2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ssv/features.py
"""Per-well preprocessing and the inputs of every candidate segment.

Coordinates. Let s be the first lattice row (chosen so that the chunk grid ends exactly at the last row) and
Z_lin the least-squares line of Z(MD) over rows >= s. The lattice coordinate of a TVT value is

    S = TVT - off,   off(row) = TVT(s) + z_det(row) - z_det(s),   z_det = Z_lin - Z.

S = 0 at row s; S constant means "layers parallel to the trajectory trend". Every candidate segment is a straight
line in S between two anchors (integers in [-KC, KC]) at consecutive chunk boundaries.

Raw inputs (before standardization, Sec. 4.2):
  sequence channels at NPE points along the extended segment line
    0 g      lateral GR (low-pass + box smoothed), global standard score
    1 t      reference GR at the TVT implied by the line, same standard score
    2 g - t  misfit
    3 dnb    S(line) - S(neighbour field)   [ft]
    4 inc    1 on the current chunk, 0 on the context chunks
    5 nan    1 where the lateral GR was missing (interpolated)
    6 dnear  median distance to the nearest labelled wells [kft]
  scalars per segment
    0 slope           move / chunk  [ft/ft]
    1 slope - field   minus the neighbour-field slope over the chunk
    2 offset          mean of dnb over the current chunk [ft]
    3 h               clipped squared misfit, mean over the current chunk (the hand-crafted cost)
    4 h_window        same over the whole window
    5 nan fraction    over the current chunk
    6 u               MD distance from the last known row [kft]
"""
import math
import numpy as np
import torch
from scipy.signal import butter, filtfilt

NCH, NSC = 7, 7
SEQ_STD = [0, 1, 2, 3, 6]          # standardized sequence channels (4, 5 are indicators)
SC_STD = [0, 1, 2, 3, 4, 6]        # standardized scalars (5 is a fraction)
NB_SEQ, NB_SC = [3, 6], [1, 2]     # neighbour-derived inputs (zeroed when the neighbour inputs are off)
_b, _a = butter(1, 0.4 / 0.5, btype="low")


def _interp_nan(x):
    x = np.asarray(x, float)
    m = np.isfinite(x)
    i = np.arange(len(x))
    if m.sum() > 1:
        return np.interp(i, i[m], x[m]), ~m
    return np.zeros(len(x)), ~m


def lowpass(x):
    return filtfilt(_b, _a, _interp_nan(x)[0])


def box(x, k):
    c = np.r_[0, np.cumsum(x)]
    i = np.arange(len(x))
    lo, hi = np.clip(i - k // 2, 0, len(x)), np.clip(i + k - k // 2, 0, len(x))
    return (c[hi] - c[lo]) / (hi - lo)


def gr_stats(wells: dict) -> tuple:
    """Global mean and std of the low-pass lateral GR over the given (training) wells."""
    a = np.concatenate([lowpass(d["GR"]) for d in wells.values()])
    return float(a.mean()), float(a.std())


def seen_reference(d, tw_t, tw_g, blend, window):
    """Reference GR(TVT): typewell, blended with the well's own known-section GR binned by its known TVT."""
    K = d["K"]
    tvi, gr = d["TVT_input"][:K], d["GR"][:K]
    m = np.isfinite(tvi) & np.isfinite(gr) & (np.abs(tvi - tvi[K - 1]) <= window)
    if m.sum() < 20:
        return tw_g
    b = np.round(tvi[m] * 4) / 4
    u, inv = np.unique(b, return_inverse=True)
    s = np.bincount(inv, gr[m]) / np.bincount(inv)
    out = tw_g.copy()
    c = (tw_t >= u.min()) & (tw_t <= u.max())
    out[c] = blend * np.interp(tw_t[c], u, s) + (1 - blend) * tw_g[c]
    return out


def lattice_start(K, N, D):
    """First lattice row s and r = K - s known rows inside the lattice (chunk grid ends at the last row)."""
    npred = N - K
    r = (D - npred % D) % D or D
    r = min(r, K - 1)
    return K - r, r


def prep(name, d, nb, cfg, gmu, gsd):
    """Numpy arrays of one well on its lattice (rows s .. s + C*D, padded with the last value)."""
    D, MV = cfg.chunk, cfg.mv
    K, N = d["K"], len(d["MD"])
    s, r = lattice_start(K, N, D)
    C = int(math.ceil((N - s) / D))
    Lr = C * D + 1
    md, Z = d["MD"], d["Z"]
    mz, cz = np.polyfit(md[s:], Z[s:], 1)
    zdet = (mz * md + cz) - Z
    tvi = d["TVT_input"][:K].astype(float)
    off_all = tvi[s] + zdet - zdet[s]
    off = off_all[s:]
    # move of the known section over the last chunk before s (lattice units) -> conditions the first move
    j0 = max(0, s - D)
    p0 = ((tvi[s] - off_all[s]) - (tvi[j0] - off_all[j0])) * D / max(s - j0, 1) if s > 0 else 0.0
    p0 = float(np.clip(p0, -3 * MV, 3 * MV))
    g, isnan = _interp_nan(d["GR"])
    g = box(filtfilt(_b, _a, g), cfg.gr_box)[s:]
    isnan = isnan[s:].astype(np.float32)
    tw_t, tw_g = d["tw_TVT"].astype(float), d["tw_GR"].astype(float)
    mt = np.isfinite(tw_t)
    tw_t, tw_g = tw_t[mt], tw_g[mt]
    o = np.argsort(tw_t)
    tw_t, tw_g = tw_t[o], tw_g[o]
    tw_g = _interp_nan(tw_g)[0] if np.isfinite(tw_g).sum() > 1 else np.full(len(tw_t), gmu)
    if cfg.seen:
        tw_g = seen_reference(d, tw_t, tw_g, cfg.seen_blend, cfg.seen_window)
    grid = np.arange(tw_t[0] - 50, tw_t[-1] + 50, cfg.tw_step)
    twg = np.interp(grid, tw_t, tw_g)

    def pad(a, val=None):
        a = np.asarray(a, np.float32)
        return np.r_[a, np.full(Lr - len(a), a[-1] if val is None else val, np.float32)]

    # neighbour field path in lattice units (known rows use the known TVT)
    nbp = np.r_[tvi[s:K], np.asarray(nb["path"], float)] - off
    nbp = _interp_nan(nbp)[0]
    dnear = np.clip(np.asarray(nb["dmed"], float)[s:] / 1000.0, 0, 3)
    y = d["TVT"][s:].astype(np.float32) if "TVT" in d else np.full(N - s, np.nan, np.float32)
    lm = np.zeros(N - s, np.float32)
    lm[r:] = 1
    lm[~np.isfinite(y)] = 0
    cs = np.arange(C)
    i0 = np.minimum(s + cs * D, N - 1)
    u = ((md[i0] - md[K - 1]) / 1000.0).astype(np.float32)
    b0, b1 = np.minimum(cs * D, len(nbp) - 1), np.minimum(cs * D + D, len(nbp) - 1)
    mnb = (nbp[b1] - nbp[b0]).astype(np.float32)                      # field move per chunk (ft)
    dnc = dnear[np.minimum(cs * D, len(dnear) - 1)].astype(np.float32)
    return dict(w=name, C=C, r=r, s=s, K=K, N=N, npred=N - K, p0=np.float32(p0),
                g=pad((g - gmu) / gsd), nan=pad(isnan, 1.0), off=pad(off), snb=pad(nbp),
                y=pad(np.nan_to_num(y), 0.0), lm=pad(lm, 0.0), twg=((twg - gmu) / gsd).astype(np.float32),
                tw0=float(grid[0]), dnear=pad(dnear, 0.0), u=u, mnb=mnb, dnc=dnc)


def to_torch(p, device):
    return {k: (torch.from_numpy(v).to(device) if isinstance(v, np.ndarray) else v) for k, v in p.items()}


class Lattice:
    """Constant tensors of the lattice for a config."""

    def __init__(self, cfg, device):
        self.D, self.KC, self.MV = cfg.chunk, cfg.kc, cfg.mv
        self.NA, self.NM = 2 * self.KC + 1, 2 * self.MV + 1
        self.NPE = cfg.pts_per_chunk * (cfg.ext_back + 1 + cfg.ext_fwd)
        self.AV = torch.arange(-self.KC, self.KC + 1, dtype=torch.float32, device=device)
        self.DV = torch.arange(-self.MV, self.MV + 1, dtype=torch.float32, device=device)
        self.REL = torch.linspace(-cfg.ext_back * self.D, (cfg.ext_fwd + 1) * self.D - 1, self.NPE, device=device)
        self.INC = ((self.REL >= 0) & (self.REL < self.D)).float()
        self.VALID = (self.AV[:, None] + self.DV[None, :]).abs() <= self.KC
        self.device = device


def raw_inputs(p, cs, lat: Lattice, clip: float, tw_step: float):
    """Raw inputs of all candidate segments of chunks cs: seq [n, NA, NM, NPE, NCH], scal [n, NA, NM, NSC]."""
    D, NA, NM = lat.D, lat.NA, lat.NM
    n = len(cs)
    Lr = len(p["g"])
    ridx = (cs[:, None] * D + lat.REL[None, :]).round().long().clamp(0, Lr - 1)
    S = (lat.AV[None, :, None, None] + lat.DV[None, None, :, None] * (lat.REL / D)[None, None, None, :])
    S = S.expand(n, -1, -1, -1)
    tv = S + p["off"][ridx][:, None, None, :]
    fi = torch.nan_to_num((tv - p["tw0"]) / tw_step, nan=0.0).clamp(0, len(p["twg"]) - 1.001)
    i0 = fi.floor().long()
    a = fi - i0
    t = p["twg"][i0] * (1 - a) + p["twg"][i0 + 1] * a

    def E(x):
        return x[ridx][:, None, None, :].expand_as(t)

    g, nn_ = E(p["g"]), E(p["nan"])
    dnb = S - p["snb"][ridx][:, None, None, :]
    inc = lat.INC[None, None, None, :].expand_as(t)
    seq = torch.stack([g, t, g - t, dnb, inc, nn_, E(p["dnear"])], -1)
    e2 = ((g - t) ** 2).clamp(max=clip)
    ninc = lat.INC.sum()
    h = (e2 * inc).sum(-1) / ninc
    hw = e2.mean(-1)
    m = (lat.DV / D)[None, None, :].expand(n, NA, -1)
    mf = (p["mnb"][cs] / D)[:, None, None].expand(-1, NA, NM)
    offn = (dnb * inc).sum(-1) / ninc
    nanf = (nn_ * inc).sum(-1) / ninc
    uc = p["u"][cs][:, None, None].expand(-1, NA, NM)
    scal = torch.stack([m, m - mf, offn, h, hw, nanf, uc], -1)
    return seq, scal


def true_path_misfit(p, lat: Lattice, tw_step: float):
    """GR misfit g - t along the TRUE path (prediction rows), for choosing the clip and for the GR analyses."""
    y, lm = p["y"], p["lm"] > 0
    fi = ((y - p["tw0"]) / tw_step).clamp(0, len(p["twg"]) - 1.001)
    i0 = fi.floor().long()
    a = fi - i0
    t = p["twg"][i0] * (1 - a) + p["twg"][i0 + 1] * a
    return (p["g"] - t)[lm]


In [ ]:
%%writefile /kaggle/working/ssv/field.py
"""Neighbour structure field: a map-view estimate of the gradient of the structural surface S = TVT + Z.

Observations are along-track dips dS/ds of labelled support wells, measured over SEG-ft pieces every SUB ft,
plus the query well's own known section (extra weight). At each query point the local gradient is a Gaussian-weighted
least-squares fit with an anisotropic distance (rotation `angle`, along-axis compression `ratio`) and a ridge pull
(`lam`) towards the regional gradient, followed by one Tukey reweighting pass (`robust`, ft/ft). Integrating the
gradient along the query trajectory gives the neighbour prediction of S, hence of TVT.
"""
import numpy as np
from scipy.spatial import cKDTree

SEG, SUB = 40, 20


def obs_from(x, y, s, rows, wid):
    i0 = rows[rows + SEG < len(x)]; i1 = i0 + SEG
    dx = x[i1] - x[i0]; dy = y[i1] - y[i0]; ds = np.hypot(dx, dy); da = s[i1] - s[i0]
    ok = (ds > 0.8 * SEG) & np.isfinite(da)
    return np.c_[(x[i0] + x[i1]) / 2, (y[i0] + y[i1]) / 2, dx / np.maximum(ds, 1e-9), dy / np.maximum(ds, 1e-9),
                 da / np.maximum(ds, 1e-9), np.full(len(i0), wid)][ok]


class Field:
    def __init__(self, W, names, angle=50.0, ratio=1.3):
        self.th = np.radians(angle); self.ratio = ratio; obs = []
        for i, w in enumerate(names):
            d = W[w]; x = d['X'].astype(float); y = d['Y'].astype(float); s = (d['TVT'] + d['Z']).astype(float)
            obs.append(obs_from(x, y, s, np.arange(0, len(x), SUB), i))
        self.O = np.vstack(obs); self.names = list(names)
        self.tree = cKDTree(self._m(self.O[:, :2]))
        self.g0 = np.linalg.lstsq(self.O[:, 2:4], self.O[:, 4], rcond=None)[0]

    def _m(self, xy):
        c, s = np.cos(self.th), np.sin(self.th)
        a = xy[:, 0] * c + xy[:, 1] * s; b = -xy[:, 0] * s + xy[:, 1] * c
        return np.c_[a / self.ratio, b]

    def path(self, d, h=900., k=128, lam=0.5, sib=None, sibw=0.2, pref=512, pref_w=3.0, idw=False, robust=0.0, step=20):
        K = d['K']; n = len(d['MD']); q = np.r_[np.arange(K - 1, n, step), n - 1]
        x = d['X'].astype(float); y = d['Y'].astype(float)
        xq = x[q]; yq = y[q]; mid = np.c_[(xq[1:] + xq[:-1]) / 2, (yq[1:] + yq[:-1]) / 2]
        kk = min(k, len(self.O)); dd, ii = self.tree.query(self._m(mid), k=kk)
        T = self.O[ii, 2:4]; g = self.O[ii, 4]
        wg = 1.0 / np.maximum(dd, 25.0) ** 2 if idw else np.exp(-0.5 * (dd / h) ** 2)
        if sib is not None and sibw != 1.0:
            wg = wg * np.where(np.isin(self.O[ii, 5].astype(int), sib), 1.0, sibw)
        if pref > 0:
            # own build-section dip observations of the query well (known TVT_input)
            s = (d['TVT_input'] + d['Z']).astype(float); r0 = max(0, K - pref)
            P = obs_from(x, y, s, np.arange(r0, K - SEG, SUB), -1)
            if len(P):
                dp = np.sqrt(((self._m(mid)[:, None, :] - self._m(P[:, :2])[None, :, :]) ** 2).sum(-1))
                wp = (1.0 / np.maximum(dp, 25.0) ** 2 if idw else np.exp(-0.5 * (dp / h) ** 2)) * pref_w
                T = np.concatenate([T, np.broadcast_to(P[None, :, 2:4], (len(mid), len(P), 2))], 1)
                g = np.concatenate([g, np.broadcast_to(P[None, :, 4], (len(mid), len(P)))], 1)
                wg = np.concatenate([wg, wp], 1)

        def solve(wg):
            Mx = np.einsum('qk,qki,qkj->qij', wg, T, T) + lam * np.eye(2)[None] * (wg.sum(1)[:, None, None] if idw else 1.0)
            b = np.einsum('qk,qki,qk->qi', wg, T, g) + lam * self.g0[None] * (wg.sum(1)[:, None] if idw else 1.0)
            return np.linalg.solve(Mx, b[..., None])[..., 0]
        Gr = solve(wg)
        if robust > 0:
            r = g - np.einsum('qki,qi->qk', T, Gr); u = np.clip(np.abs(r) / robust, 0, 1)
            Gr = solve(wg * (1 - u ** 2) ** 2 + 1e-12)
        dS = np.r_[0, np.cumsum(Gr[:, 0] * np.diff(xq) + Gr[:, 1] * np.diff(yq))]; z = d['Z'][q].astype(float)
        tv = float(d['TVT_input'][K - 1]) + dS - (z - z[0])
        return np.interp(np.arange(K, n), q, tv), float(np.median(dd[:, 0]))


In [ ]:
%%writefile /kaggle/working/ssv/model.py
"""Segment-cost network, transition (momentum) network, standardization, lattice cost assembly and decoding."""
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from .dp import BIG, forward_backward, viterbi
from .features import NCH, NSC, SEQ_STD, SC_STD, NB_SEQ, NB_SC, raw_inputs, true_path_misfit, Lattice

H_SCALAR = 3   # index of the hand-crafted misfit among the scalars


@torch.no_grad()
def fit_stats(ps, lat: Lattice, cfg, max_chunks=6, seed=0):
    """Standardization constants from training wells.

    clip: squared GR misfit is clipped at (99th percentile of |g - t| along the TRUE paths)^2.
    mean / std of every input over all valid candidate segments of up to `max_chunks` random chunks per well.
    """
    rs = np.random.RandomState(seed)
    r = torch.cat([true_path_misfit(p, lat, cfg.tw_step) for p in ps]).abs()
    q = torch.quantile(r[torch.randperm(len(r))[:2_000_000]], torch.tensor([0.5, 0.9, 0.99], device=r.device))
    clip = float(q[2]) ** 2 if cfg.misfit_clip < 0 else cfg.misfit_clip
    s1 = torch.zeros(NCH, dtype=torch.float64, device=lat.device)
    s2 = torch.zeros_like(s1)
    c1 = torch.zeros(NSC, dtype=torch.float64, device=lat.device)
    c2 = torch.zeros_like(c1)
    n_seq = n_sc = 0
    for p in ps:
        cs = torch.from_numpy(rs.choice(p["C"], min(max_chunks, p["C"]), replace=False)).to(lat.device)
        seq, scal = raw_inputs(p, cs, lat, clip, cfg.tw_step)
        v = lat.VALID[None].expand(len(cs), -1, -1)
        sq, sc = seq[v].double(), scal[v].double()
        s1 += sq.sum((0, 1)); s2 += (sq ** 2).sum((0, 1)); n_seq += sq.shape[0] * sq.shape[1]
        c1 += sc.sum(0); c2 += (sc ** 2).sum(0); n_sc += sc.shape[0]
    mu, sd = s1 / n_seq, (s2 / n_seq - (s1 / n_seq) ** 2).clamp_min(1e-12).sqrt()
    cmu, csd = c1 / n_sc, (c2 / n_sc - (c1 / n_sc) ** 2).clamp_min(1e-12).sqrt()
    if cfg.scaling == "legacy":     # earlier version: fixed scales, offset /10 ft, misfit clipped at 25 (5 GR std)
        mu, cmu = torch.zeros_like(mu), torch.zeros_like(cmu)
        sd = torch.tensor([1, 1, 1, 10, 1, 1, 1], dtype=torch.float64)
        csd = torch.tensor([1 / 20, 1 / 20, 20, 1, 1, 1, 5], dtype=torch.float64)
        clip = 25.0
    return dict(clip=clip, misfit_q50_q90_q99=[float(x) for x in q], seq_mu=mu.tolist(), seq_sd=sd.tolist(),
                sc_mu=cmu.tolist(), sc_sd=csd.tolist())


class SSVNet(nn.Module):
    def __init__(self, cfg, stats):
        super().__init__()
        self.cfg, self.clip = cfg, stats["clip"]
        self.MV = cfg.mv
        mu, sd = torch.zeros(NCH), torch.ones(NCH)
        mu[SEQ_STD] = torch.tensor(stats["seq_mu"], dtype=torch.float32)[SEQ_STD]
        sd[SEQ_STD] = torch.tensor(stats["seq_sd"], dtype=torch.float32)[SEQ_STD]
        cmu, csd = torch.zeros(NSC), torch.ones(NSC)
        cmu[SC_STD] = torch.tensor(stats["sc_mu"], dtype=torch.float32)[SC_STD]
        csd[SC_STD] = torch.tensor(stats["sc_sd"], dtype=torch.float32)[SC_STD]
        for k, v in dict(seq_mu=mu, seq_sd=sd, sc_mu=cmu, sc_sd=csd).items():
            self.register_buffer(k, v)
        keep_seq, keep_sc = torch.ones(NCH), torch.ones(NSC)
        if not cfg.position:
            keep_sc[6] = 0
        self.register_buffer("seq_keep_nb", keep_seq.clone())
        self.register_buffer("sc_keep_nb", keep_sc.clone())
        keep_seq[NB_SEQ] = 0
        keep_sc[NB_SC] = 0
        self.register_buffer("seq_keep_no", keep_seq)
        self.register_buffer("sc_keep_no", keep_sc)
        h, M = cfg.hidden, cfg.mlp
        self.conv = nn.Sequential(nn.Conv1d(NCH, h, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
                                  nn.Conv1d(h, h, 3, padding=1), nn.ReLU())
        self.mlp = nn.Sequential(nn.Linear(2 * h + NSC, M), nn.ReLU())
        self.head = nn.Linear(M + NSC, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)
        with torch.no_grad():                       # start exactly at the (standardized) hand-crafted misfit cost
            self.head.weight[0, M + H_SCALAR] = 1.0
        self.tau = nn.Sequential(nn.Linear(9, 32), nn.ReLU(), nn.Linear(32, 1))
        nn.init.zeros_(self.tau[2].weight)
        nn.init.zeros_(self.tau[2].bias)
        self.logT = nn.Parameter(torch.tensor(float(cfg.logT0)))

    def segment_cost(self, seq, scal, nbon):
        """seq [N, NPE, NCH], scal [N, NSC] raw inputs; nbon scalar 0/1 (neighbour inputs on)."""
        seq = (seq - self.seq_mu) / self.seq_sd
        scal = (scal - self.sc_mu) / self.sc_sd
        on = float(self.cfg.neighbour and nbon)
        seq = seq * (self.seq_keep_nb if on else self.seq_keep_no)
        scal = scal * (self.sc_keep_nb if on else self.sc_keep_no)
        z = self.conv(seq.transpose(1, 2))
        z = torch.cat([z.mean(2), z.amax(2), scal], 1)
        return self.head(torch.cat([self.mlp(z), scal], 1))[:, 0]

    def transition(self, p, m, u, mnb, dn, mnbp):
        """Momentum cost of move m after move p (lattice units); mnb, mnbp: field moves of this / previous chunk."""
        MV = self.MV
        x = torch.broadcast_tensors(p / MV, m / MV, (m - p) / MV, u / 5, mnb / MV, (m - mnb) / MV, dn,
                                    (p - mnbp) / MV, (m - mnb - 0.6 * (p - mnbp)) / MV)
        return self.tau(torch.stack(x, -1))[..., 0]


def well_costs(net, p, lat, block=24):
    """Segment costs [C, NA, NM] of one well (BIG for moves leaving the lattice)."""
    out = []
    for c0 in range(0, p["C"], block):
        cs = torch.arange(c0, min(p["C"], c0 + block), device=lat.device)

        def f(cs=cs):
            seq, scal = raw_inputs(p, cs, lat, net.clip, net.cfg.tw_step)
            n = len(cs)
            v = net.segment_cost(seq.reshape(-1, lat.NPE, NCH), scal.reshape(-1, NSC), p.get("nbon", 1))
            v = v.reshape(n, lat.NA, lat.NM)
            return torch.where(lat.VALID[None], v, torch.full_like(v, BIG))
        out.append(checkpoint(f, use_reentrant=False) if torch.is_grad_enabled() else f())
    return torch.cat(out, 0)


def batch_costs(net, ps, lat):
    """Stack the lattice costs of several wells (padded with zero-cost straight moves)."""
    Cm = max(p["C"] for p in ps)
    dev = lat.device
    pad = torch.full((lat.NA, lat.NM), BIG, device=dev)
    pad[:, lat.MV] = 0.0
    cost = torch.stack([torch.cat([well_costs(net, p, lat), pad[None].expand(Cm - p["C"], -1, -1)], 0) for p in ps])
    if not net.cfg.momentum:
        return cost, torch.zeros(len(ps), Cm, lat.NM, lat.NM, device=dev), torch.zeros(len(ps), lat.NM, device=dev)
    P_, M_ = lat.DV[:, None], lat.DV[None, :]
    taus, tau0 = [], []
    for p in ps:
        on = float(net.cfg.neighbour and p.get("nbon", 1))
        ext = Cm - p["C"]
        u = torch.cat([p["u"], p["u"][-1:].expand(ext)]) * float(net.cfg.position)
        mnb = torch.cat([p["mnb"], torch.zeros(ext, device=dev)]) * on
        dn = torch.cat([p["dnc"], p["dnc"][-1:].expand(ext)]) * on
        mnbp = torch.cat([mnb[:1], mnb[:-1]])
        tt = net.transition(P_[None], M_[None], u[:, None, None], mnb[:, None, None], dn[:, None, None],
                            mnbp[:, None, None])
        live = torch.arange(Cm, device=dev) < p["C"]
        taus.append(torch.where(live[:, None, None], tt, torch.zeros_like(tt)))
        tau0.append(net.transition(torch.tensor(float(p["p0"]), device=dev), lat.DV, u[0], mnb[0], dn[0], mnb[0]))
    return cost, torch.stack(taus), torch.stack(tau0)


def rows_tvt(Sb, ps, lat):
    """Anchor values at the boundaries [B, C+1] -> per-row TVT on each well's lattice rows."""
    B, Cp1 = Sb.shape
    f = torch.arange(lat.D, device=Sb.device, dtype=Sb.dtype) / lat.D
    Srow = torch.cat([(Sb[:, :-1, None] + (Sb[:, 1:] - Sb[:, :-1])[:, :, None] * f).reshape(B, -1), Sb[:, -1:]], 1)
    return [Srow[i, :len(p["off"])] + p["off"] for i, p in enumerate(ps)]


def decode(net, ps, lat, hard=False):
    cost, tau, tau0 = batch_costs(net, ps, lat)
    if hard:
        Sb = viterbi(cost, tau, tau0, lat.KC, lat.MV)
    else:
        Sb = forward_backward(cost, tau, tau0, net.logT.exp(), lat.KC, lat.MV)
    return rows_tvt(Sb, ps, lat)


def prediction_rows(t, p):
    """Model rows of the prediction zone (rows K .. N-1 of the well)."""
    return t[p["r"]:p["r"] + p["npred"]]


In [ ]:
%%writefile /kaggle/working/ssv/neighbours.py
"""Fold-safe neighbour inputs: the structure-field path and the distance to the nearest labelled wells.

For a training well the support set is the wells of the other folds; for a test well it is all training wells.
"""
import os
import pickle
import time
import numpy as np
from scipy.spatial import cKDTree
from .field import Field
from .data import folds


def build(targets: dict, support_wells: dict, every=20, k_near=16) -> dict:
    """targets/support_wells: {name: well dict}. Returns {name: {'path': TVT along the well, 'dmed': ft}}."""
    names = sorted(support_wells)
    fld = Field(support_wells, names)
    pts = np.vstack([np.c_[support_wells[w]["X"][::every], support_wells[w]["Y"][::every]] for w in names])
    tree = cKDTree(pts)
    out = {}
    for w, d in targets.items():
        dd, _ = tree.query(np.c_[d["X"], d["Y"]], k=k_near)
        out[w] = dict(path=fld.path(d, robust=0.05)[0].astype(np.float32),
                      dmed=np.median(dd, 1).astype(np.float32))
    return out


def train_cache(wells: dict, path: str, n_folds=5, seed=42) -> dict:
    """Neighbour inputs for every training well, each built from the other folds only (cached on disk)."""
    if path and os.path.exists(path):
        return pickle.load(open(path, "rb"))
    names = sorted(wells)
    f = folds(names, n_folds, seed)
    out, t0 = {}, time.time()
    for k in range(n_folds):
        sup = {w: wells[w] for w, fk in zip(names, f) if fk != k}
        tgt = {w: wells[w] for w, fk in zip(names, f) if fk == k}
        out.update(build(tgt, sup))
        print(f"neighbour cache fold {k}: {time.time() - t0:.0f}s", flush=True)
    if path:
        pickle.dump(out, open(path, "wb"))
    return out


In [ ]:
%%writefile /kaggle/working/ssv/predict.py
"""Predict the test wells with one or more trained models (average of their soft paths) and write submission.csv.

    python -m ssv.predict --models runs/f0s0/model.pt runs/f1s0/model.pt ... --out submission.csv
"""
import argparse
import os
import numpy as np
import pandas as pd
import torch
from .config import Config
from .data import load_split, data_dir
from .features import prep, to_torch, Lattice
from .model import SSVNet, decode, prediction_rows
from . import neighbours


def load_model(path, device):
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = Config.from_dict(ck["config"])
    net = SSVNet(cfg, ck["stats"]).to(device)
    net.load_state_dict(ck["state"])
    net.eval()
    return net, cfg, ck["stats"]


@torch.no_grad()
def predict_wells(net, cfg, stats, wells, nb, device, batch=4, hard=False):
    lat = Lattice(cfg, device)
    out, names = {}, sorted(wells)
    for i in range(0, len(names), batch):
        ps = [to_torch(prep(w, wells[w], nb[w], cfg, stats["gr_mu"], stats["gr_sd"]), device) for w in names[i:i + batch]]
        for t, p in zip(decode(net, ps, lat, hard), ps):
            out[p["w"]] = prediction_rows(t, p).cpu().numpy()
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--out", default="submission.csv")
    ap.add_argument("--data", default=None)
    ap.add_argument("--hard", action="store_true", help="Viterbi path instead of the posterior mean")
    a = ap.parse_args(argv)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    train, test = load_split("train", a.data), load_split("test", a.data)
    nb = neighbours.build(test, train)              # test wells: support = all training wells
    preds = []
    for m in a.models:
        net, cfg, stats = load_model(m, dev)
        preds.append(predict_wells(net, cfg, stats, test, nb, dev, hard=a.hard))
        print("predicted with", m, flush=True)
    rows = {}
    for w, d in test.items():
        t = np.mean([p[w] for p in preds], 0)
        assert len(t) == len(d["MD"]) - d["K"]
        for j, v in enumerate(t):
            rows[f"{w}_{d['K'] + j}"] = float(v)
    sub = pd.read_csv(os.path.join(a.data or data_dir(), "sample_submission.csv"))
    col = [c for c in sub.columns if c != "id"][0]
    sub[col] = sub["id"].map(rows)
    missing = int(sub[col].isna().sum())
    print(f"rows {len(sub)}, missing {missing}", flush=True)
    assert missing == 0, "every id of the sample submission must be predicted"
    sub.to_csv(a.out, index=False)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ssv/train.py
"""Train one model on one fold (or on all wells with fold = -1).

    python -m ssv.train --config '{"fold": 0, "seed": 0}' --out runs/f0s0 [--nb-cache nb.pkl]

Writes <out>/model.pt (EMA weights + config + standardization constants), <out>/log.jsonl (one line per epoch),
and <out>/val.pkl (prediction-zone soft and hard TVT of every validation well).
"""
import argparse
import copy
import json
import math
import os
import pickle
import time
import numpy as np
import torch
from .config import Config
from .data import load_split, split_names
from .features import prep, to_torch, gr_stats, Lattice
from .model import SSVNet, fit_stats, batch_costs, rows_tvt, decode, prediction_rows
from .dp import forward_backward
from . import neighbours


def batch_loss(pred, ps, norm, rows_per_well):
    se = sum(((t - p["y"]) ** 2 * p["lm"]).sum() for t, p in zip(pred, ps))
    n = sum(p["lm"].sum() for p in ps)
    if norm == "global":          # each predicted row has the same weight in every batch
        return se / (len(ps) * rows_per_well), se, n
    if norm == "batch":           # rows of short-well batches weigh more
        return se / n, se, n
    if norm == "well":            # each well has the same weight (rows of short wells weigh more)
        return sum(((t - p["y"]) ** 2 * p["lm"]).sum() / p["lm"].sum() for t, p in zip(pred, ps)) / len(ps), se, n
    raise ValueError(norm)


@torch.no_grad()
def evaluate(net, P, names, lat, B, keep=False):
    net.eval()
    se = dict(soft=0.0, hard=0.0)
    n = 0.0
    out = {}
    for i in range(0, len(names), B):
        ps = [P[w] for w in names[i:i + B]]
        for k, hard in (("soft", False), ("hard", True)):
            pred = decode(net, ps, lat, hard)
            for t, p in zip(pred, ps):
                se[k] += float(((t - p["y"]) ** 2 * p["lm"]).sum())
                if keep:
                    out.setdefault(p["w"], {})[k] = prediction_rows(t, p).cpu().numpy()
        n += float(sum(p["lm"].sum() for p in ps))
    r = {k: round(math.sqrt(v / n), 4) for k, v in se.items()}
    return (r, out) if keep else r


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="{}", help="JSON string or path to a JSON file")
    ap.add_argument("--out", required=True)
    ap.add_argument("--nb-cache", default="nb_train.pkl")
    ap.add_argument("--data", default=None)
    ap.add_argument("--limit", type=int, default=0, help="debug: use only this many train and val wells")
    a = ap.parse_args(argv)
    cfg = Config.from_json(open(a.config).read() if os.path.exists(a.config) else a.config)
    os.makedirs(a.out, exist_ok=True)
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    W = load_split("train", a.data)
    tr, va = split_names(W, cfg.fold, cfg.n_folds, cfg.fold_seed)
    if a.limit:
        tr, va = tr[:a.limit], va[:max(1, a.limit // 4)]
    NB = neighbours.train_cache(W, a.nb_cache, cfg.n_folds, cfg.fold_seed)
    gmu, gsd = gr_stats({w: W[w] for w in tr})
    t0 = time.time()
    P = {w: to_torch(prep(w, W[w], NB[w], cfg, gmu, gsd), dev) for w in tr + va}
    lat = Lattice(cfg, dev)
    stats = fit_stats([P[w] for w in tr], lat, cfg, seed=cfg.seed)
    stats.update(gr_mu=gmu, gr_sd=gsd)
    net = SSVNet(cfg, stats).to(dev)
    print(json.dumps(dict(event="start", wells_train=len(tr), wells_val=len(va), device=dev,
                          prep_s=round(time.time() - t0, 1), params=sum(q.numel() for q in net.parameters()),
                          config=json.loads(cfg.to_json()), stats=stats)), flush=True)
    ema = copy.deepcopy(net)
    opt = torch.optim.AdamW(net.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    B = cfg.batch_wells
    steps = cfg.epochs * math.ceil(len(tr) / B)
    rows_per_well = float(np.mean([float(P[w]["lm"].sum()) for w in tr]))
    rs = np.random.RandomState(cfg.seed)
    log = open(os.path.join(a.out, "log.jsonl"), "w")
    if va:
        r0 = evaluate(net, P, va, lat, B)
        log.write(json.dumps(dict(epoch=-1, val=r0)) + "\n"); log.flush()
        print(json.dumps(dict(epoch=-1, val=r0)), flush=True)
    st = 0
    for ep in range(cfg.epochs):
        net.train()
        t1 = time.time()
        se_sum = n_sum = 0.0
        order = list(rs.permutation(tr))
        for i in range(0, len(order), B):
            ps = []
            for w in order[i:i + B]:
                p = dict(P[w])
                p["nbon"] = 0 if (cfg.nb_dropout > 0 and rs.rand() < cfg.nb_dropout) else 1
                ps.append(p)
            for g_ in opt.param_groups:
                g_["lr"] = cfg.lr * (cfg.lr_min_frac + (1 - cfg.lr_min_frac) * 0.5 * (1 + math.cos(math.pi * st / steps)))
            cost, tau, tau0 = batch_costs(net, ps, lat)
            Sb = forward_backward(cost, tau, tau0, net.logT.exp(), lat.KC, lat.MV)
            loss, se, n = batch_loss(rows_tvt(Sb, ps, lat), ps, cfg.loss_norm, rows_per_well)
            opt.zero_grad()
            loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.grad_clip)
            opt.step()
            st += 1
            with torch.no_grad():
                for e, q in zip(ema.parameters(), net.parameters()):
                    e.mul_(cfg.ema).add_(q, alpha=1 - cfg.ema)
            se_sum += float(se.detach())
            n_sum += float(n)
        rec = dict(epoch=ep, train_rmse=round(math.sqrt(se_sum / n_sum), 4), T=round(float(net.logT.exp()), 5),
                   lr=opt.param_groups[0]["lr"], sec=round(time.time() - t1, 1))
        if va:
            rec["val"] = evaluate(ema, P, va, lat, B)
        log.write(json.dumps(rec) + "\n"); log.flush()
        print(json.dumps(rec), flush=True)
        torch.save(dict(state=ema.state_dict(), config=json.loads(cfg.to_json()), stats=stats),
                   os.path.join(a.out, "model.pt"))
    if va:
        r, kp = evaluate(ema, P, va, lat, B, keep=True)
        pickle.dump(kp, open(os.path.join(a.out, "val.pkl"), "wb"))
        print(json.dumps(dict(event="final", val=r)), flush=True)
    log.close()


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, os, json, time, sys
os.chdir('/kaggle/working')
JOBS = [
 {
  "tag": "legacy_f0_s0",
  "config": {
   "scaling": "legacy",
   "fold": 0,
   "seed": 0,
   "tag": "legacy"
  }
 },
 {
  "tag": "legacy_f1_s0",
  "config": {
   "scaling": "legacy",
   "fold": 1,
   "seed": 0,
   "tag": "legacy"
  }
 },
 {
  "tag": "legacy_f2_s0",
  "config": {
   "scaling": "legacy",
   "fold": 2,
   "seed": 0,
   "tag": "legacy"
  }
 },
 {
  "tag": "legacy_f3_s0",
  "config": {
   "scaling": "legacy",
   "fold": 3,
   "seed": 0,
   "tag": "legacy"
  }
 },
 {
  "tag": "legacy_f4_s0",
  "config": {
   "scaling": "legacy",
   "fold": 4,
   "seed": 0,
   "tag": "legacy"
  }
 },
 {
  "tag": "batchloss_f0_s0",
  "config": {
   "loss_norm": "batch",
   "fold": 0,
   "seed": 0,
   "tag": "batchloss"
  }
 },
 {
  "tag": "batchloss_f1_s0",
  "config": {
   "loss_norm": "batch",
   "fold": 1,
   "seed": 0,
   "tag": "batchloss"
  }
 },
 {
  "tag": "batchloss_f2_s0",
  "config": {
   "loss_norm": "batch",
   "fold": 2,
   "seed": 0,
   "tag": "batchloss"
  }
 },
 {
  "tag": "batchloss_f3_s0",
  "config": {
   "loss_norm": "batch",
   "fold": 3,
   "seed": 0,
   "tag": "batchloss"
  }
 },
 {
  "tag": "batchloss_f4_s0",
  "config": {
   "loss_norm": "batch",
   "fold": 4,
   "seed": 0,
   "tag": "batchloss"
  }
 }
]
import torch
ngpu = max(1, torch.cuda.device_count())
# fold-safe neighbour inputs for every training well (CPU, ~1 min)
subprocess.run([sys.executable, '-c', 'from ssv.data import load_split; from ssv import neighbours; '
                'neighbours.train_cache(load_split("train"), "/kaggle/working/nb_train.pkl")'], check=True)
running, queue = {}, list(JOBS)
logs = {}
while queue or running:
    for g in range(ngpu):
        if g not in running and queue:
            job = queue.pop(0)
            out = f"/kaggle/working/runs/{job['tag']}"; os.makedirs(out, exist_ok=True)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), OMP_NUM_THREADS='2')
            f = open(out + '/stdout.txt', 'w')
            running[g] = (job, subprocess.Popen([sys.executable, '-m', 'ssv.train', '--config', json.dumps(job['config']),
                                                 '--out', out, '--nb-cache', '/kaggle/working/nb_train.pkl'],
                                                env=env, stdout=f, stderr=subprocess.STDOUT), time.time(), f)
            print('started', job['tag'], 'on GPU', g, flush=True)
    time.sleep(30)
    for g, (job, p, t0, f) in list(running.items()):
        if p.poll() is not None:
            f.close(); del running[g]
            tail = open(f"/kaggle/working/runs/{job['tag']}/stdout.txt").read().splitlines()[-3:]
            print('finished', job['tag'], 'exit', p.returncode, f'{(time.time()-t0)/3600:.2f} h', flush=True)
            for l in tail: print('   ', l[:400], flush=True)
    for g, (job, p, t0, f) in running.items():
        lp = f"/kaggle/working/runs/{job['tag']}/log.jsonl"
        if os.path.exists(lp):
            ls = open(lp).read().splitlines()
            if ls and logs.get(job['tag']) != ls[-1]:
                logs[job['tag']] = ls[-1]; print(job['tag'], ls[-1][:300], flush=True)
